# Moldova VACS — PUD exploration (2013 release path)

Single file **`MOLDOVA_VACS_2013_PUD.dta`** in **`data/raw/Moldova Stata/`**. **`pyreadstat.read_dta`** succeeds with **default** encoding here (retry **`latin1`** if you hit decode errors on another machine).

**Codebook PDF** (`MOLDOVA_VACS_2013_Codebook.pdf`): use for **design narrative** (PSU sampling, **STRATA** value labels, **REG** as “Region”, **SAMPLEWEIGHT** / **NTOT** definitions). Page headers sometimes say **“Survey Year: 2019”** while the folder/file name says **2013**—treat the **Data User Guide** + file naming as authoritative for the wave label you use in Excel.

This PUD includes a released string **`PUD_ID`** (unique). **Household:** no separate `hh` column; VACS selects one adolescent per household, so one row ≈ one household.

**Flow:** §1 Load → §2 column list & EDA → §3 samples & slot summaries → §4 harmonized TSV.


In [1]:
from pathlib import Path

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
import re

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

COUNTRY_DIR = ROOT / "data" / "raw" / "Moldova Stata"
PUD_PATH = COUNTRY_DIR / "MOLDOVA_VACS_2013_PUD.dta"
READ_KW = {}  # use {"encoding": "latin1"} if read_dta fails on strings

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")


## 1. Load data

`read_dta(PUD_PATH, **READ_KW)`.


In [2]:
if not PUD_PATH.is_file():
    raise FileNotFoundError(f"Expected:\n  {PUD_PATH}")

df, meta = pyreadstat.read_dta(PUD_PATH, **READ_KW)
print(f"File: {PUD_PATH}")
print(f"Rows × columns: {df.shape[0]:,} × {df.shape[1]:,}")

df = df.copy()

if "sex" in df.columns:
    print("sex (1 / 2; confirm male/female in codebook):")
    display(df["sex"].value_counts(dropna=False).sort_index())

assert "PUD_ID" in df.columns
print(f"PUD_ID unique / rows: {df['PUD_ID'].nunique():,} / {len(df):,}")
print(f"PUD_ID duplicated: {int(df['PUD_ID'].duplicated().sum())}")
print(f"Max rows per psu: {int(df.groupby('psu').size().max())}")

_chk = 0
for pid, psu in zip(df["PUD_ID"], df["psu"]):
    a, b, c = pid.rsplit("_", 2)
    if int(b) != psu:
        _chk += 1
print(f"PUD_ID tail parse vs psu mismatches: {_chk}")

print(f"Duplicate rows (all columns): {int(df.duplicated().sum())}")

df[["PUD_ID", "sex", "reg", "loc", "psu", "strata", "SampleWeight", "ntot"]].head(4)


File: /Users/starsrain/research_side_projects_ipv/data/raw/Moldova Stata/MOLDOVA_VACS_2013_PUD.dta
Rows × columns: 2,002 × 504
sex (1 / 2; confirm male/female in codebook):


sex
1     978
2    1024
Name: count, dtype: int64

PUD_ID unique / rows: 2,002 / 2,002
PUD_ID duplicated: 0
Max rows per psu: 22
PUD_ID tail parse vs psu mismatches: 0
Duplicate rows (all columns): 0


,PUD_ID,sex,reg,loc,psu,strata,SampleWeight,ntot
0,1_Center - Rural_3_1,1,Anenii,Bulboaca,3,Center - Rural,288.614601,3
1,1_Center - Rural_3_2,1,Anenii,Bulboaca,3,Center - Rural,144.307301,4
2,1_Center - Rural_3_3,1,Anenii,Bulboaca,3,Center - Rural,144.307301,2
3,1_Center - Rural_3_5,1,Anenii,Bulboaca,3,Center - Rural,170.431253,5


## 2. Column list & quick EDA


In [3]:
name_to_label = dict(meta.column_names_to_labels) if meta.column_names_to_labels else {}

var_table = pd.DataFrame({
    "column": df.columns,
    "stata_label": [name_to_label.get(c, "") or "" for c in df.columns],
    "dtype": df.dtypes.astype(str).values,
    "missing_n": df.isna().sum().values,
    "missing_pct": (100 * df.isna().mean()).round(2),
})
print(f"Variables: {len(df.columns):,}  |  Observations: {len(df):,}")
display(var_table.head(40))
display(var_table.sort_values("missing_pct", ascending=False).head(15).reset_index(drop=True))
df.info(max_cols=18)


Variables: 504  |  Observations: 2,002


,column,stata_label,dtype,missing_n,missing_pct
sex,sex,Sex,int64,0,0.00
reg,reg,,str,0,0.00
loc,loc,,str,0,0.00
psu,psu,psu,int64,0,0.00
ntot,ntot,,int64,0,0.00
H2,H2,,str,0,0.00
H3,H3,,object,240,11.99
H4,H4,,object,239,11.94
H5,H5,,object,239,11.94
H6,H6,,object,254,12.69


,column,stata_label,dtype,missing_n,missing_pct
0,Q505,,object,2001,99.95
1,Q507,,object,2001,99.95
2,Q908,,object,1997,99.75
3,Q907,,object,1997,99.75
4,Q906,,object,1997,99.75
5,Q905,,object,1997,99.75
6,Q904,,object,1997,99.75
7,Q903,,object,1997,99.75
8,Q902,,object,1997,99.75
9,Q100C1,,object,1994,99.60


<class 'pandas.DataFrame'>
RangeIndex: 2002 entries, 0 to 2001
Columns: 504 entries, sex to PUD_ID
dtypes: float64(1), int64(97), object(352), str(54)
memory usage: 7.7+ MB


## 3. Further EDA and exploration

### Raw row samples


In [ ]:
pd.set_option("display.max_columns", 42)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 72)

_core = [c for c in [
    "PUD_ID", "sex", "reg", "loc", "psu", "strata",
    "SampleWeight", "ntot", "H3",
] if c in df.columns]
sub = df[_core]
display(sub.head(8))
display(sub.sample(6, random_state=0))


### Slot summaries (ID / geo / design)


In [4]:
L = meta.column_names_to_labels or {}

_WORD_SEX = re.compile(r"\b(?:male|females?|female)\b", re.IGNORECASE)
_EMBED_MF = re.compile(r"(?i)(?:^|_)(?:male|female)(?=_|$)")


def _abstract_digit_pattern(val: str) -> str:
    parts = []
    i = 0
    while i < len(val):
        ch = val[i]
        if ch.isdigit():
            j = i
            while j < len(val) and val[j].isdigit():
                j += 1
            parts.append("N" * (j - i))
            i = j
        elif ch.isalpha():
            j = i
            while j < len(val) and val[j].isalpha():
                j += 1
            parts.append("A")
            i = j
        else:
            parts.append(ch)
            i += 1
    return "".join(parts)


def _unified_style_pattern(st: pd.Series):
    st = st.dropna().astype(str)
    if len(st) == 0:
        return None
    abstracts = st.map(_abstract_digit_pattern)
    if abstracts.nunique(dropna=False) != 1:
        return None
    pat = abstracts.iloc[0]
    if not any(ch.isdigit() for ch in pat):
        return None
    if set(pat) <= {"N"}:
        return None
    ex = st.iloc[0]
    if len(pat) > 72:
        return f"{pat[:72]}… (e.g. {ex[:40]}{'…' if len(ex) > 40 else ''})"
    return f"{pat} (e.g. {ex})"


def _width_note(s: pd.Series) -> str:
    sn = s.dropna()
    if len(sn) == 0:
        return "n/a"
    if pd.api.types.is_numeric_dtype(s):
        whole = (sn == sn.astype(float).astype(int)).all()
        if whole:
            lens = sn.astype(int).astype(str).str.len()
            lo, hi = int(lens.min()), int(lens.max())
            return f"{lo}-{hi} digits (integer codes)" if lo != hi else f"{lo} digits (integer codes)"
        lens = sn.astype(str).str.len()
        lo, hi = int(lens.min()), int(lens.max())
        return f"{lo}-{hi} chars (numeric as string)" if lo != hi else f"{lo} chars (numeric as string)"
    st = sn.astype(str)
    lens = st.str.len()
    lo, hi = int(lens.min()), int(lens.max())
    w = f"{lo}-{hi} chars" if lo != hi else f"{lo} chars"
    if st.str.fullmatch(r"\d+").all():
        return f"{w} (string; all numeric characters)"
    return f"{w} (string)"


def _special_id_note(s: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(s):
        return "no M/F identifier (numeric)"
    st = s.dropna().astype(str)
    if len(st) == 0:
        return "n/a"
    if st.str.contains(_WORD_SEX, regex=True, na=False).any() or st.str.contains(_EMBED_MF, regex=True, na=False).any():
        return "Male/Female text (words or _Female_/_Male_ segments)"
    return "no male/female text (heuristic)"


def slot_summary(title: str, cols: list, note_extra: str = ""):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    miss = [c for c in cols if c not in df.columns]
    if miss:
        print("MISSING columns:", miss)
        return
    for c in cols:
        s = df[c]
        lbl = (str(L.get(c) or ""))[:75]
        size_part = _width_note(s)
        style = _unified_style_pattern(s) if not pd.api.types.is_numeric_dtype(s) else None
        style_part = f"; style {style}" if style else ""
        id_part = _special_id_note(s)
        if pd.api.types.is_numeric_dtype(s):
            sn = s.dropna()
            extra = f"min/max={sn.min()}/{sn.max()}" if len(sn) else "min/max=n/a"
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}; {extra}"
        else:
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}"
        print(f"  {c} | {lbl}")
        print(f"    {info}; n_distinct={s.nunique(dropna=True)}; missing={s.isna().sum()}")
    if note_extra:
        print("  ", note_extra)


slot_summary("1. Respondent ID (released)", ["PUD_ID"], "always 3 underscores: {sex}_{strata}_{psu}_{within-psu index}; rsplit(_,2) recovers psu")
slot_summary("2. Region (codebook: REG)", ["reg"], "abbreviated district/region names in PUD (36 levels)")
slot_summary("3. Locality", ["loc"], "")
slot_summary("4. Stratum (codebook: STRATA / Selection_Strata)", ["strata"], "12 labeled design strata")
slot_summary("5. Cluster / PSU", ["psu"], "integer PSU id; codebook describes PSU-based sample")
slot_summary("6. Sex", ["sex"], "")
slot_summary("7. Weight", ["SampleWeight"], "no HIV* weight columns in this PUD")
slot_summary("8. NTOT (household size)", ["ntot"], "codebook: how many people in household")



1. Respondent ID (released)
  PUD_ID | 
    20-42 chars (string); no male/female text (heuristic); dtype=str; n_distinct=2002; missing=0
   always 3 underscores: {sex}_{strata}_{psu}_{within-psu index}; rsplit(_,2) recovers psu

2. Region (codebook: REG)
  reg | 
    5-6 chars (string); no male/female text (heuristic); dtype=str; n_distinct=36; missing=0
   abbreviated district/region names in PUD (36 levels)

3. Locality
  loc | 
    5-10 chars (string); no male/female text (heuristic); dtype=str; n_distinct=132; missing=0

4. Stratum (codebook: STRATA / Selection_Strata)
  strata | Selection_Strata
    12-32 chars (string); no male/female text (heuristic); dtype=str; n_distinct=12; missing=0
   12 labeled design strata

5. Cluster / PSU
  psu | psu
    1-4 digits (integer codes); no M/F identifier (numeric); dtype=int64; min/max=3/3061; n_distinct=185; missing=0
   integer PSU id; codebook describes PSU-based sample

6. Sex
  sex | Sex
    1 digits (integer codes); no M/F identifier

## 4. Harmonized codebook slots (Moldova — PUD path 2013)

**Single combined PUD** — **`variable_male`** and **`variable_female`** repeat the same names.

**Source:** `data/raw/Moldova Stata/MOLDOVA_VACS_2013_PUD.dta`.

**Codebook / guide:** `MOLDOVA_VACS_2013_Codebook.pdf` (variable definitions, strata labels, **SAMPLEWEIGHT**); **`MOLDOVA_VACS_2013_DataUserGuide.pdf`** for sample design and weighting narrative.

### Household

There is **no** separate household key. Under VACS (one sampled adolescent per household), **each row is one household**; **`PUD_ID`** is the practical row identifier.

```
slot	variable_male	variable_female	type_and_width	notes
Respondent ID	PUD_ID	PUD_ID	**20–42 chars** (string); exactly **3 underscores** per value	Released ID; structure **`{sex}_{strata text}_{psu}_{rank_within_psu}`**; **`rsplit('_', 2)`** recovers **`psu`** and rank; leading **`{sex}_`…** matches **`sex`** + **`strata`** columns
Household ID	—	—	—	**No `hh` column**; one row per household interview; use **`PUD_ID`** for row linkage
Geo level 1	reg	reg	5–6 chars (string) abbrev region names	Codebook: **REG** — Region (full names in PDF; PUD uses short spellings e.g. Anenii, Chisin)
Geo level 2	loc	loc	5–10 chars (string)	Locality / loc (132 distinct in file)
Geo level 3	—	—	—	Not separately released
Stratum	strata	strata	string; 12 labels (e.g. Center - Rural, Chisinau - Chisinau City - Urban)	Codebook **STRATA** / Stata label **Selection_Strata**; combines macro-region + urban/rural design cells
Cluster / PSU	psu	psu	integer; 1–4 digit string form (3–3061 here; 185 distinct)	PSU; use with **strata** + **SampleWeight** for `svy` (confirm in Data User Guide)
Sex	sex	sex	1 / 2 (integer)	Match to codebook coding (male/female)
Weight	SampleWeight	SampleWeight	float (~28.8–1246.6 in file)	Codebook: **Final Sample Weight**; **no** `HIVWeight` columns in this extract
Interview date	—	—	—	No **date** / **interview** labels on columns in a quick pass—confirm in questionnaires / guide if needed
```

**Excel:** Note the **2013 vs 2019** wording mismatch inside some PDF pages if your harmonized **survey year** must be exact.
